# From an activated conda instance open jupyter notebook to Analyze Xenium data

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib seaborn scanpy squidpy anndata harmonypy leidenalg -q

In [ ]:
import warnings

warnings.filterwarnings("ignore")  # scanpy/anndata/squidpy chatter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import squidpy as sq
import anndata as ad
print(f"squidpy=={sq.__version__}")

Download the Feature-cell Matrix (HDF5) and the Cell summary file (CSV).
You need these 2 files in a new folder:
- `cell_feature_matrix.h5`
- `cells.csv`

(note the cells.csv file is compressed and accessible via `gunzip filename.csv.gz`)

In [ ]:
import os

DATA_DIR = '/home/workspace/Xenium/Data'
os.chdir(DATA_DIR)
print(f"Working directory set to: {os.getcwd()}")

In [ ]:
sk1 = sc.read_h5ad('TIS05393.h5ad')
sk2 = sc.read_h5ad('TIS04788.h5ad')
sk3 = sc.read_h5ad('TIS05392.h5ad')
sk4 = sc.read_h5ad('TIS05394.h5ad')
sk5 = sc.read_h5ad('TIS04789.h5ad')
sk6 = sc.read_h5ad('TIS05397.h5ad')

sk1.obs_names_make_unique()
sk2.obs_names_make_unique()
sk3.obs_names_make_unique()
sk4.obs_names_make_unique()
sk5.obs_names_make_unique()
sk6.obs_names_make_unique()

combined_adatas = sc.concat({
    'TIS05393': sk1,
    'TIS04788': sk2,
    'TIS05392': sk3,
    'TIS05394': sk4,
    'TIS04789': sk5,
    'TIS05397': sk6
}, label='dataset_name', join='outer')
combined_adatas.obs_names_make_unique()

In [ ]:
print(f'Number of observations (cells): {combined_adatas.n_obs}')
print(f'Number of variables (genes): {combined_adatas.n_vars}')

In [ ]:
print(combined_adatas.obs['dataset_name'].value_counts())

In [ ]:
import matplotlib as mpl
import scipy.sparse as sp

combined_adatas.layers["counts"] = combined_adatas.X.copy()
sc.pp.normalize_total(combined_adatas, inplace=True)
# sc.pp.log1p(combined_adatas)  # Uncomment to log-transform

# Ensure X stays sparse after normalization
if not sp.issparse(combined_adatas.X):
    combined_adatas.X = sp.csr_matrix(combined_adatas.X)

sc.pp.pca(combined_adatas, n_comps=30, use_highly_variable=False)
sc.pp.neighbors(combined_adatas, n_neighbors=15, n_pcs=30)
sc.tl.umap(combined_adatas)
sc.tl.leiden(combined_adatas)

In [ ]:
sc.pl.umap(
    combined_adatas,
    color=["total_counts", "n_genes_by_counts", "leiden", 'dataset_name'],
    wspace=0.4,
)

In [ ]:
# Harmony batch correction (bypassing scanpy wrapper due to version mismatch)
import harmonypy as hm

ho = hm.run_harmony(combined_adatas.obsm['X_pca'].astype('float64'), combined_adatas.obs, 'dataset_name')

# Handle shape depending on harmonypy version (Z_corr may be n_pcs x n_cells or n_cells x n_pcs)
z = ho.Z_corr
if z.shape[0] != combined_adatas.n_obs:
    z = z.T
combined_adatas.obsm['X_pca_harmony'] = z
combined_adatas.obsm['X_pca'] = combined_adatas.obsm['X_pca_harmony']

sc.pp.neighbors(combined_adatas, use_rep='X_pca')
sc.tl.umap(combined_adatas)
sc.tl.leiden(combined_adatas, flavor='igraph', n_iterations=2, directed=False)

In [ ]:
plt.rcParams['savefig.dpi'] = 300

sc.pl.umap(
    combined_adatas,
    color=["total_counts", "n_genes_by_counts", "leiden", 'dataset_name'],
    wspace=0.4,
    save='_UMAP_MM.png'
)

sc.tl.leiden(combined_adatas, flavor='igraph', n_iterations=2, directed=False, resolution=0.8)
sc.tl.rank_genes_groups(combined_adatas, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(combined_adatas, n_genes=10, sharey=False, fontsize=20)

In [ ]:
cluster_to_cell_type = {
    '0': 'Erythroblast_1',
    '1': 'Macrophage',
    '2': 'Monocyte',
    '3': 'Endothelial_1',
    '4': 'Monocyte/Macrophage',
    '5': 'Stromal/Fibroblast',
    '6': 'Tumor-Associated Macrophage',
    '7': 'Neoplasm_1',
    '8': 'Erythroblast_2',
    '9': 'Neutrophil',
    '10': 'Neoplasm_2',
    '11': 'Neoplasm_3',
    '12': 'Cytotoxic T Cell',
    '13': 'Neoplasm_4',
    '14': 'Myeloid',
    '15': 'Neoplasm_5',
    '16': 'Endothelial_2',
}

combined_adatas.obs['cell_type'] = combined_adatas.obs['leiden'].map(cluster_to_cell_type)

In [ ]:
combined_adatas.obs['cell_type'] = combined_adatas.obs['cell_type'].astype('category')
print(combined_adatas.obs['cell_type'].cat.categories)

In [ ]:
plt.rcParams['savefig.dpi'] = 300

plt.figure(figsize=(12, 8))
sc.pl.umap(
    combined_adatas,
    color='cell_type',
    legend_loc='right margin',
    legend_fontsize=8,
    show=False,
    save='_UMAP_celltypes_only.png'
)
plt.savefig('/home/workspace/Xenium/Data/umap_celltypes_only.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
slide_ids = combined_adatas.obs['dataset_name'].unique()
genes_to_plot = ['MZB1', 'PECAM1', 'GNLY']

cmap = mpl.colors.ListedColormap(['cyan', 'magenta', 'orange', 'red']).with_extremes(under='0.8', over='red')
bounds = [3, 4, 8, 20]
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

fig, axes = plt.subplots(len(slide_ids), len(genes_to_plot), figsize=(5 * len(genes_to_plot), 5 * len(slide_ids)))

for i, slide_id in enumerate(slide_ids):
    slide_data = combined_adatas[combined_adatas.obs['dataset_name'] == slide_id]
    for j, gene in enumerate(genes_to_plot):
        ax = axes[i, j] if len(slide_ids) > 1 else axes[j]
        sc.pl.spatial(slide_data, color=gene, ax=ax, spot_size=50, cmap=cmap, norm=norm, show=False)
        if i == 0:
            ax.set_title(gene)
        if j == 0:
            ax.set_ylabel(f'Slide {slide_id}')

plt.tight_layout()
plt.show()

In [ ]:
def split_umap(combined_adatas, split_by, ncol=2, nrow=None, save_path='umap_MM_plots.png', **kwargs):
    categories = combined_adatas.obs[split_by].cat.categories
    if nrow is None:
        nrow = int(np.ceil(len(categories) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(5*ncol, 4*nrow), dpi=300)
    axs = axs.flatten()
    for i, cat in enumerate(categories):
        ax = axs[i]
        sq.pl.spatial_scatter(
            combined_adatas[combined_adatas.obs[split_by] == cat],
            library_id="spatial",
            shape=None,
            color=["leiden"],
            wspace=0.4,
            ax=ax,
            title=cat
        )
    plt.tight_layout()
    plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')


split_umap(combined_adatas=combined_adatas, split_by='dataset_name', ncol=3, save_path='umap_by_MM_name.png')